# Master Synthesis: P2 Xenium 空间转录组多维度重建与科学评估总览

**任务标识**：`p2-xenium-impact-master-synthesis`  
**核心宗旨**：跳出单纯的工程环境跑通，**从底层科学机制与空间生物学需求出发，全面评估不同空间转录组重建机制（Raw 基线、Random 随机对照、Within-Cluster OT 空间传输、Outside-Cluster OT 传输、Mean 均值）对组织空间结构、微环境表型、全转录组功能及空间细胞通讯的真实影响。**

---

## 一、 顶层全局对比大矩阵（Master Evaluation Matrix）

本矩阵系统对比 5 种状态在 5 个核心科学维度上的表现、机理与生物学意义：

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# 构造顶层主评估对比矩阵
matrix_data = [
    {
        "科学评估维度": "1. 单细胞表达特征与保真度",
        "原生测量 (Raw Baseline)": "422 探针原位实测；\n中位深度: 84~112 counts；\n无参考转移偏差",
        "随机基线 (Random Baseline)": "13,088 基因离散替换；\nDonor 复用率 80.2% (最大 34)；\n中位深度: 35~56 counts",
        "簇内最优传输 (Within-Cluster OT)": "13,088 基因连续加权传输；\n消除离散克隆复制；\n空间连续概率场",
        "跨簇最优传输 (Outside-Cluster OT)": "13,088 基因全局连续传输；\n允许大类内跨亚型流变；\n全局宏观连续平滑",
        "簇均值对照 (Mean Reference)": "13,088 基因离散阶跃常数；\n簇内细胞完全均质同质化；\n微观方差被强制置零",
        "机制洞悉与核心结论": "Random 属于离散 donor 拷贝，无法反映微环境渐变；OT 方法转为加权连续流形，恢复连续分布。",
        "关联专项": "NB-01 重建表达对比"
    },
    {
        "科学评估维度": "2. 空间连续性与自相关 (Moran's I)",
        "原生测量 (Raw Baseline)": "原位物理自相关基线；\n反映局部微环境生理调控；\n探针密度限制了检测宽度",
        "随机基线 (Random Baseline)": "普遍显著暴跌 (>90% 基因负差)；\n中位数差 -0.02 ~ -0.05；\n注入离散单细胞高频白噪声",
        "簇内最优传输 (Within-Cluster OT)": "预期显著反弹回升；\n借助 ot_smoothing 空间平滑先验；\n消除空间椒盐噪声",
        "跨簇最优传输 (Outside-Cluster OT)": "大尺度组织平滑；\n保留长程解剖梯度；\n需防范微观过度平滑",
        "簇均值对照 (Mean Reference)": "簇内极高相关 (人造同质)；\n簇边界出现阶跃断崖；\n丧失单细胞真实波动",
        "机制洞悉与核心结论": "Random 的 Moran 暴跌符合其白噪声机制预期；OT+空间平滑能修复物理邻域平滑度。",
        "关联专项": "NB-02 空间自相关与平滑"
    },
    {
        "科学评估维度": "3. 局部表型多样性与 Gain",
        "原生测量 (Raw Baseline)": "Raw Leiden 默认低分辨率；\n受限于靶向基因表达谱；\n簇数较小 (K=5~10)",
        "随机基线 (Random Baseline)": "原生呈现虚假正 Gain；\nK-control 对齐后翻转为负；\n完全依赖聚类数不对称伪影",
        "簇内最优传输 (Within-Cluster OT)": "基于空间图的真实精细亚型；\nK-control 下检验微环境共存；\n局部相分离度 (Mixing Index)",
        "跨簇最优传输 (Outside-Cluster OT)": "刻画中间态连续流变；\n解析跨亚型分化过渡区；\n表征浸润前沿多态性",
        "簇均值对照 (Mean Reference)": "状态完全退化为固定常数；\n有效状态数严重阶跃化；\n无法表征微观亚态多样性",
        "机制洞悉与核心结论": "香农熵 Neff 受全局聚类数 K 影响极大；必须结合 K-control 与空间邻域纯度评估真实增益。",
        "关联专项": "NB-03 表型多样性与 Gain"
    },
    {
        "科学评估维度": "4. 全转录组扩增与空间互作 (CCI)",
        "原生测量 (Raw Baseline)": "仅测 422 探针；\nEMT 仅 31/200 基因；\n配受体对严重残缺，无法计算 CCI",
        "随机基线 (Random Baseline)": "扩至 13,088 基因；\nAUCell 出现 83% 死零截断；\n白噪声稀释了配受体空间共定位",
        "簇内最优传输 (Within-Cluster OT)": "激活配受体对空间邻近富集；\n恢复微环境旁分泌通讯；\n连续型评分刻画通路梯度",
        "跨簇最优传输 (Outside-Cluster OT)": "刻画长程趋化因子互作轴；\n捕获肿瘤-间质大尺度通讯；\n通路评分连续渐变",
        "簇均值对照 (Mean Reference)": "粗粒度簇间通信；\n掩盖单细胞水平特异性互作；\n通路打分退化为阶跃常数",
        "机制洞悉与核心结论": "全转录组扩增最大价值是激活 CellPhoneDB 空间互作；连续型打分解决 AUCell 死零截断。",
        "关联专项": "NB-04 全转录组与空间互作"
    },
    {
        "科学评估维度": "5. 解剖微区与病理边界 (Anatomy)",
        "原生测量 (Raw Baseline)": "含 Tumor Core、Normal 与 Interface；\n具有完整病理解剖架构；\n但缺乏未测基因微观细节",
        "随机基线 (Random Baseline)": "缺失肿瘤与上皮细胞 (100% Other)；\n单子类型硬编码跳过所致；\n无法评估病理微环境边界",
        "簇内最优传输 (Within-Cluster OT)": "单子类型直通修复后：\n恢复全组织解剖边界；\n特异性定位肿瘤界面浸润亚群",
        "跨簇最优传输 (Outside-Cluster OT)": "单子类型直通修复后：\n表征肿瘤向间质渗透的渐变梯度；\n解析侵袭前沿微环境",
        "簇均值对照 (Mean Reference)": "粗粒度硬切割解剖区域；\n边界处出现不自然的数值断层；\n缺乏浸润渐变",
        "机制洞悉与核心结论": "必须通过 singleton passthrough 恢复全组织覆盖，解剖微区与侵袭边界分析才具备生物学基础。",
        "关联专项": "NB-01/04 联合覆盖"
    }
]

df_matrix = pd.DataFrame(matrix_data)
pd.set_option('display.max_colwidth', None)
display(df_matrix)


## 二、 核心科学事实与深入机制归因（Key Findings & Attribution）

回顾并审视首轮真实 P2CRC Xenium 数据运行（iter-001 原生、iter-002 K-control 诊断、共同基因诊断）所沉淀的确定性科学事实：

In [ ]:
from pathlib import Path
import json

# 提取首轮已固化的真实实验数据事实
findings_summary = [
    {
        "事实编号": "Fact-01: Moran's I 暴跌",
        "实测证据": "在 343 共同基因轴下，All 298/343 负差、Fib 307/333 负差、Mono 315/338 负差、T 305/339 负差；差值中位数 -0.02 ~ -0.05。",
        "机制本质": "Random Donor 离散抽样整行替换实测表达，破坏了原位微环境物理连续性，向空间图中注入了高频随机白噪声。",
        "解决方案": "转向带空间平滑 (ot_smoothing) 的最优传输连续加权重建 (Within-Cluster / Outside-Cluster OT)。"
    },
    {
        "事实编号": "Fact-02: K-control 翻转 Gain",
        "实测证据": "原生 Gain 中位数 All +0.65、Fib +0.016；对齐 Raw 簇数后，All 暴跌至 +0.016，Fib 翻转为 0 (均值 -0.155)，T 翻转为 -0.245。",
        "机制本质": "局域稀释香农熵 Neff 受全局聚类数 K 影响极大；原生正增益纯属类别基数不对称 (K=31 vs 10) 的统计伪影。",
        "解决方案": "放弃对香农熵绝对值的过度依赖，引入物理空间邻域纯度 (Local Purity) 与相分离度 (Mixing Index) 综合度量。"
    },
    {
        "事实编号": "Fact-03: AUCell 死零截断",
        "实测证据": "在 343 共同基因下，Raw 由于小 panel 检出低推导出 rank_cutoff=2，导致 24,898 / 30,000 (83%) 细胞得分为绝对零值。",
        "机制本质": "AUCell 强行套用于极小靶向 panel 产生的病态截断，使得两侧 EMT 得分脱离生物学真实尺度。",
        "解决方案": "采用连续加权分布打分 (Continuous Scanpy score_genes 或 Pseudobulk Z-score) 替代离散死零截断。"
    },
    {
        "事实编号": "Fact-04: Anatomy 100% 降级",
        "实测证据": "交付的 50,626 个 SVC 细胞全为间质与免疫细胞，所有物理网格 100% 判定为 Other，Tumor/Interface 完全丢失。",
        "机制本质": "单细胞参考集中 Tumor 与上皮细胞缺少二级子类型，被代码 subtype_count <= 1 硬编码规则静默跳过 (丢失 16.5 万细胞)。",
        "解决方案": "在生产端实现单子类型直通通道 (Singleton Passthrough)，恢复占组织 80% 的肿瘤与上皮细胞全景覆盖。"
    }
]

display(pd.DataFrame(findings_summary))


## 三、 专项证明 Notebook 矩阵导航与职责划分

为彻底证明上述机制并验证改进方案，后续 4 个专项 Notebook 的分工如下：

1. **`01_reconstruction_methods_benchmark.ipynb`**：
   * 横向对比 `random` vs `within_cluster` vs `outside_cluster` vs `mean` 4 种模式；
   * 统计 Donor 复用率消除情况、单细胞测序稀疏度及计算收敛耗时。
2. **`02_spatial_autocorrelation_and_smoothness.ipynb`**：
   * 检验最优传输（OT + `ot_smoothing`）能否有效遏制白噪声，恢复 Global Moran's I 与微环境 LISA 局域热点。
3. **`03_microenvironment_diversity_and_gain.ipynb`**：
   * 在受控聚类粒度与连续空间邻域纯度下，评估重建后微环境精细亚型的真实共存态。
4. **`04_pathway_and_spatial_communication.ipynb`**：
   * 激活 CellPhoneDB 空间互作（CCI），评估全转录组扩增在恢复原位未测配受体通讯对上的真实生物学价值。
